# 01 — Data Inspection

Mục tiêu: kiểm tra 5 bảng raw trước khi Join. **Chưa làm sạch hay mô hình hóa ở notebook này.**

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_loader import RAW_FILES, load_all_tables, inspect_tables

In [2]:
tables = load_all_tables(ROOT / "data" / "raw")
inspection = inspect_tables(tables)
inspection

,table,rows,columns,duplicate_rows,missing_cells
0,demographics,7043,9,0,0
1,location,7043,10,0,0
2,population,1671,3,0,0
3,services,7043,31,0,5403
4,status,7043,12,0,10348


In [3]:
for name, df in tables.items():
    print(f"\n=== {name.upper()} ===")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    display(df.head(3))


=== DEMOGRAPHICS ===
shape: (7043, 9)
columns: ['Customer ID', 'Count', 'Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents']


,Customer ID,Count,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents
0,8779-QRDMV,1,Male,78,No,Yes,No,No,0
1,7495-OOKFY,1,Female,74,No,Yes,Yes,Yes,1
2,1658-BYGOY,1,Male,71,No,Yes,No,Yes,3



=== LOCATION ===
shape: (7043, 10)
columns: ['Location ID', 'Customer ID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude']


,Location ID,Customer ID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude
0,OXCZEW7397,8779-QRDMV,1,United States,California,Los Angeles,90022,"34.02381, -118.156582",34.023810,-118.156582
1,FCCECI8494,7495-OOKFY,1,United States,California,Los Angeles,90063,"34.044271, -118.185237",34.044271,-118.185237
2,HEHUQY7254,1658-BYGOY,1,United States,California,Los Angeles,90065,"34.108833, -118.229715",34.108833,-118.229715



=== POPULATION ===
shape: (1671, 3)
columns: ['ID', 'Zip Code', 'Population']


,ID,Zip Code,Population
0,1,90001,54492
1,2,90002,44586
2,3,90003,58198



=== SERVICES ===
shape: (7043, 31)
columns: ['Service ID', 'Customer ID', 'Count', 'Quarter', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue']


,Service ID,Customer ID,Count,Quarter,Referred a Friend,Number of Referrals,Tenure in Months,Offer,Phone Service,Avg Monthly Long Distance Charges,...,Unlimited Data,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue
0,IJKDQVSWH3522,8779-QRDMV,1,Q3,No,0,1,NaN,No,0.00,...,No,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,59.65
1,BFKMZJAIE2285,7495-OOKFY,1,Q3,Yes,1,8,Offer E,Yes,48.85,...,Yes,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,1024.10
2,EIMVJQBMT7187,1658-BYGOY,1,Q3,No,0,18,Offer D,Yes,11.33,...,Yes,Month-to-Month,Yes,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,1910.88



=== STATUS ===
shape: (7043, 12)
columns: ['Status ID', 'Customer ID', 'Count', 'Quarter', 'Satisfaction Score', 'Customer Status', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Category', 'Churn Reason']


,Status ID,Customer ID,Count,Quarter,Satisfaction Score,Customer Status,Churn Label,Churn Value,Churn Score,CLTV,Churn Category,Churn Reason
0,SWSORB1252,8779-QRDMV,1,Q3,3,Churned,Yes,1,91,5433,Competitor,Competitor offered more data
1,SNAEQA8572,7495-OOKFY,1,Q3,3,Churned,Yes,1,69,5302,Competitor,Competitor made better offer
2,LMBQNN3714,1658-BYGOY,1,Q3,2,Churned,Yes,1,81,3179,Competitor,Competitor made better offer


## Kiểm tra khóa CustomerID

Các bảng khách hàng phải có `Customer ID` và không được có duplicate key trước khi dùng `validate="one_to_one"`.

In [4]:
from src.features import normalize_customer_id

for name in ["demographics", "location", "services", "status"]:
    clean = normalize_customer_id(tables[name])
    print(name, "rows=", len(clean), "unique CustomerID=", clean["CustomerID"].nunique(), "duplicates=", clean["CustomerID"].duplicated().sum())

demographics rows= 7043 unique CustomerID= 7043 duplicates= 0
location rows= 7043 unique CustomerID= 7043 duplicates= 0
services rows= 7043 unique CustomerID= 7043 duplicates= 0
status rows= 7043 unique CustomerID= 7043 duplicates= 0


# Nhận xét: 
- Trùng khớp quan hệ 1-1: 4 bảng (demographics, location, services, status) đều chứa đúng 7,043 dòng tương ứng với 7,043 CustomerID duy nhất và không bị lặp (duplicates = 0). Điều này đảm bảo an toàn khi thực hiện merge với thiết lập validate="one_to_one".   
- Bảng địa lý: Bảng population chứa thông tin dân số tương ứng với 1,671 mã Zip Code độc lập.   

## Missing values

Tập trung kiểm tra `Total Charges` vì dữ liệu nguồn có thể biểu diễn giá trị trống dưới dạng chuỗi/whitespace.

In [5]:
for name, df in tables.items():
    miss = df.isna().sum().sort_values(ascending=False)
    print(f"\n{name}")
    display(miss[miss.gt(0)].head(15))


demographics


Series([], dtype: int64)


location


Series([], dtype: int64)


population


Series([], dtype: int64)


services


Offer            3877
Internet Type    1526
dtype: int64


status


Churn Reason      5174
Churn Category    5174
dtype: int64

# Nhận xét:
- Các bảng demographics, location, và population hoàn toàn không có dữ liệu khuyết.
- Chi tiết các cột khuyết dữ liệu ở 2 bảng còn lại:
+ Bảng services:
Offer: khuyết 3,877 giá trị.   
Internet Type: khuyết 1,526 giá trị.   
+ Bảng status:
Churn Category: khuyết 5,174 giá trị.   
Churn Reason: khuyết 5,174 giá trị. 


### Kết luận trước khi sang bước 02

Toàn bộ các bảng thô đạt yêu cầu về cardinalities và tính duy nhất của khóa chính CustomerID. Cấu trúc dữ liệu đáp ứng đủ điều kiện để tiến hành bước tiếp theo (02 — Data Cleaning & Feature Engineering).